# CCK confound sweep: chunker=words400
Runs confound_sweep.py from the committed repo at pinned SHA 3cfa6f8. Internet ON, CPU, attach the competition. Stage 1 is a tiny smoke test; stage 2 is the real run. Each config writes its JSON when done.


In [ ]:
!pip install fastembed transformers numpy --quiet
print('deps installed')


In [ ]:
from fastembed import TextEmbedding
from transformers import AutoTokenizer
TextEmbedding('BAAI/bge-small-en-v1.5')
AutoTokenizer.from_pretrained('BAAI/bge-small-en-v1.5')
AutoTokenizer.from_pretrained('unsloth/gemma-2-2b-bnb-4bit')
print('all cached', flush=True)


In [ ]:
import subprocess, glob
SHA='3cfa6f893558c63ebcb4d82494ab89f01d294c69'
def sh(c):
    r=subprocess.run(c,shell=True,capture_output=True,text=True); print(c,'->',r.returncode,r.stderr.strip()[-200:])
sh('git clone https://github.com/pupa3066/context-configuration-builder.git /tmp/cck')
sh(f'cd /tmp/cck && git checkout {SHA}')
for name,url in [('requests','https://github.com/psf/requests.git'),('click','https://github.com/pallets/click.git'),('black','https://github.com/psf/black.git')]:
    sh(f'git clone --depth=200 {url} /tmp/{name}')
    print(name, len(glob.glob(f'/tmp/{name}/**/*.py',recursive=True)), '.py files')


In [ ]:
import subprocess, sys, time
REPOS='requests:/tmp/requests click:/tmp/click black:/tmp/black'
def stream(cmd):
    with subprocess.Popen(cmd,shell=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as p:
        for line in p.stdout:
            if 'SyntaxWarning' in line or line.strip().startswith('<unknown>'): continue
            print(line,end='',flush=True)
    return p.returncode
base=f'cd /tmp/cck && {sys.executable} -u benchmark/confound_sweep.py --repos {REPOS} --chunker words400'
print('--- SMOKE', time.strftime('%H:%M:%S'), flush=True)
rc=stream(base+' --n-per-repo 3 --caps 200 --bootstrap 50 --out-dir /tmp/smoke')
assert rc==0, 'smoke test failed - stop here and send the error'
print('--- REAL', time.strftime('%H:%M:%S'), flush=True)
rc=stream(base+' --n-per-repo 80 --caps 1000 100000 --bootstrap 1000 --out-dir /kaggle/working')
print('done exit',rc,time.strftime('%H:%M:%S'))
